# Train a compatible CIFAR-10 ResNet-18

This notebook trains the repository's custom CIFAR-10 `resnet18` and saves a checkpoint that can be used by `train-wa.py` with `--pretrained-file clean-resnet18`.

In [ ]:
import os
import sys

REPO_DIR = "/content/csds440-code-for-colab"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/cmk237/csds440-code-for-colab.git {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Colab: Runtime > Change runtime type > T4 GPU")

In [ ]:
!mkdir -p ./data/cifar10

# Download the archive once
!wget -c https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz \
    -O ./data/cifar-10-python.tar.gz

# Extract so the final path is ./data/cifar10/cifar-10-batches-py/
!tar -xzf ./data/cifar-10-python.tar.gz -C ./data/cifar10

# Verify the expected files exist
!ls ./data/cifar10/cifar-10-batches-py

In [ ]:
import torch
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

from core.data import get_data_info
from core.models import create_model

device = torch.device("cuda")
batch_size = 512
epochs = 50
learning_rate = 0.1
num_workers = 2
checkpoint_dir = "./logs/clean-resnet18"
os.makedirs(checkpoint_dir, exist_ok=True)

mean = [0.4914, 0.4822, 0.4465]
std = [0.2023, 0.1994, 0.2010]
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])
test_transform = transforms.ToTensor()

train_dataset = torchvision.datasets.CIFAR10(
    root="./data/cifar10", train=True, download=False, transform=train_transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root="./data/cifar10", train=False, download=False, transform=test_transform
)
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=True,
)

info = get_data_info("./data/cifar10")
model = create_model("resnet18", normalize=True, info=info, device=device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(), lr=learning_rate, momentum=0.9,
    weight_decay=5e-4, nesterov=True,
)
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[25, 40], gamma=0.1
)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Batches per epoch:", len(train_loader))

In [ ]:
best_accuracy = 0.0

for epoch in range(1, epochs + 1):
    model.train()
    train_correct = 0
    train_total = 0
    train_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        train_total += labels.size(0)

    scheduler.step()

    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images)
            test_correct += (outputs.argmax(dim=1) == labels).sum().item()
            test_total += labels.size(0)

    train_accuracy = 100.0 * train_correct / train_total
    test_accuracy = 100.0 * test_correct / test_total
    print(
        f"Epoch {epoch:03d}/{epochs} | "
        f"loss {train_loss / train_total:.4f} | "
        f"train {train_accuracy:.2f}% | test {test_accuracy:.2f}%"
    )

    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "test_accuracy": test_accuracy,
                "epoch": epoch,
                "model": "resnet18",
                "normalize": True,
            },
            os.path.join(checkpoint_dir, "weights-best.pt"),
        )

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "test_accuracy": test_accuracy,
        "epoch": epochs,
        "model": "resnet18",
        "normalize": True,
    },
    os.path.join(checkpoint_dir, "weights-last.pt"),
)
print(f"Best test accuracy: {best_accuracy:.2f}%")
print(f"Saved best checkpoint: {checkpoint_dir}/weights-best.pt")

In [ ]:
checkpoint = torch.load(
    "./logs/clean-resnet18/weights-best.pt",
    map_location=device,
)
print(checkpoint.keys())
print("Saved test accuracy:", checkpoint["test_accuracy"])
print("Checkpoint is ready for train-wa.py")

Use the checkpoint for adversarial training with:

```bash
!python train-wa.py \
    --data-dir ./data \
    --log-dir ./logs \
    --desc adv-from-clean-resnet18 \
    --data cifar10s \
    --model resnet18 \
    --normalize true \
    --pretrained-file clean-resnet18 \
    --beta 6.0 \
    --aux-data-filename ./data/auxiliary/cifar10_ddpm_relabeled.npz
```

The checkpoint is saved at `./logs/clean-resnet18/weights-best.pt`, which is the location expected by the repository's `--pretrained-file` option.